# Charging Station Scene Labeling via OSM POI

基于 OpenStreetMap POI 数据，为每个充电站打上场景标签（对应 NTS 出行目的分类，精简为 9 类）。

**精简标签对应关系（NTS → 9类）：**

| 标签 | 对应 NTS 原始分类 |
|---|---|
| `home` | Home, Escort home |
| `work` | Work, In course of work, Escort work, Escort in course of work |
| `education` | Education, Escort education |
| `shopping` | Food shopping, Non food shopping, Escort shopping/personal business |
| `personal_business` | Personal business medical/eat-drink/other |
| `social` | Eat/drink with friends, Visit friends, Other social |
| `leisure` | Entertain/public activity, Sport: participate, Day trip/just walk |
| `holiday` | Holiday: base |
| `other` | Other non-escort, Other escort |

**方法**：对每个充电站坐标做 500m 缓冲区查询，统计周边各类 OSM POI 数量，按加权得分决定标签。

## 0. 依赖安装

```bash
pip install osmnx geopandas shapely pandas tqdm pyarrow
```

In [1]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

ox.settings.log_console = False
ox.settings.use_cache = True   # 缓存 OSM 查询，避免重复请求

## 1. 加载充电站数据

In [2]:
df = pd.read_csv('../UK_OCM_stations.csv')
df = df.dropna(subset=['Latitude', 'Longitude']).reset_index(drop=True)
print(f'充电站总数: {len(df)}')
df.head()

充电站总数: 26959


,StationID,Latitude,Longitude,Title,TotalCapacity_kW,StationType,Bands
0,194240,53.478011,-2.228680,NaN,2442.0,Fast site,Fast (8–49kW)
1,194241,55.952561,-3.183221,NaN,1848.0,Fast site,Fast (8–49kW)
2,155229,56.004720,-3.753830,NaN,1419.0,Fast site,Fast (8–49kW);Rapid (50–149kW)
3,256446,51.577544,-0.223825,NaN,1320.0,Fast site,Fast (8–49kW)
4,194034,51.459956,-0.970644,NaN,1106.0,Fast site,Slow (3–8kW)


## 2. 定义 POI 查询标签与权重

格式：`{scene: [(osm_key, osm_value, weight), ...]}`  
权重越高表示该 POI 对该场景越有代表性。

In [ ]:
SCENE_TAGS = {
    # ── home ──────────────────────────────────────────────────────────────
    # NTS: Home, Escort home
    'home': [
        ('landuse',  'residential', 3),
        ('building', 'residential', 2),
        ('building', 'apartments',  2),
        ('building', 'house',       2),
        ('building', 'detached',    1),
        ('building', 'terrace',     1),
    ],

    # ── work ──────────────────────────────────────────────────────────────
    # NTS: Work, In course of work, Escort work, Escort in course of work
    'work': [
        ('landuse',  'commercial',  3),
        ('landuse',  'industrial',  2),
        ('building', 'office',      3),
        ('building', 'commercial',  2),
        ('building', 'industrial',  2),
        ('office',   True,          2),
        ('amenity',  'workplace',   2),
    ],

    # ── education ─────────────────────────────────────────────────────────
    # NTS: Education, Escort education
    'education': [
        ('amenity',  'school',       3),
        ('amenity',  'university',   3),
        ('amenity',  'college',      3),
        ('amenity',  'kindergarten', 2),
        ('landuse',  'education',    3),
        ('building', 'school',       2),
        ('building', 'university',   2),
    ],

    # ── shopping ──────────────────────────────────────────────────────────
    # NTS: Food shopping, Non food shopping, Escort shopping/personal business
    'shopping': [
        ('shop',     True,           3),
        ('landuse',  'retail',       3),
        ('building', 'retail',       2),
        ('building', 'supermarket',  2),
        ('amenity',  'marketplace',  2),
        ('amenity',  'food_court',   1),
    ],

    # ── personal_business ─────────────────────────────────────────────────
    # NTS: Personal business medical, Personal business eat/drink, Personal business other
    'personal_business': [
        ('amenity',  'hospital',      3),
        ('amenity',  'clinic',        3),
        ('amenity',  'doctors',       3),
        ('amenity',  'dentist',       2),
        ('amenity',  'pharmacy',      2),
        ('amenity',  'bank',          2),
        ('amenity',  'post_office',   1),
        ('amenity',  'restaurant',    1),
        ('amenity',  'cafe',          1),
        ('amenity',  'fast_food',     1),
        ('amenity',  'bar',           1),
    ],

    # ── social ────────────────────────────────────────────────────────────
    # NTS: Eat/drink with friends, Visit friends, Other social
    'social': [
        ('amenity',  'pub',           2),
        ('amenity',  'bar',           2),
        ('amenity',  'nightclub',     2),
        ('amenity',  'community_centre', 2),
        ('amenity',  'social_facility',  2),
        ('amenity',  'place_of_worship', 1),
        ('building', 'community_centre', 1),
    ],

    # ── leisure ───────────────────────────────────────────────────────────
    # NTS: Entertain/public activity, Sport: participate, Day trip/just walk
    'leisure': [
        ('leisure',  True,                 3),
        ('amenity',  'cinema',             2),
        ('amenity',  'theatre',            2),
        ('amenity',  'sports_centre',      2),
        ('amenity',  'swimming_pool',      2),
        ('sport',    True,                 2),
        ('landuse',  'recreation_ground',  2),
        ('landuse',  'grass',              1),
        ('natural',  'park',               1),
        ('tourism',  'attraction',         1),
    ],

    # ── holiday ───────────────────────────────────────────────────────────
    # NTS: Holiday: base
    'holiday': [
        ('tourism',  'hotel',         3),
        ('tourism',  'motel',         3),
        ('tourism',  'guest_house',   3),
        ('tourism',  'hostel',        2),
        ('tourism',  'camp_site',     2),
        ('tourism',  'caravan_site',  2),
        ('tourism',  'attraction',    1),
        ('tourism',  'museum',        1),
    ],

    # ── other ─────────────────────────────────────────────────────────────
    # NTS: Other non-escort, Other escort (fallback — no dominant POI signal)
    # (will only be assigned when all scores are tied at 0 → 'unknown' preferred)
}

RADIUS_M = 500  # 查询半径（米）

## 3. 构建统一查询标签

将所有场景的 OSM 标签合并为一个 `QUERY_TAGS` 字典，后续批量下载只需请求一次。

In [ ]:
def _build_query_tags():
    """合并所有场景的 OSM 标签，生成一次性查询参数。"""
    all_tags = {}
    for tag_list in SCENE_TAGS.values():
        for key, value, _ in tag_list:
            if key not in all_tags:
                all_tags[key] = set()
            if value is True:
                all_tags[key] = True
            elif all_tags[key] is not True:
                all_tags[key].add(value)
    for k in list(all_tags):
        if isinstance(all_tags[k], set):
            vals = list(all_tags[k])
            all_tags[k] = vals[0] if len(vals) == 1 else vals
    return all_tags

QUERY_TAGS = _build_query_tags()
print('OSM 查询键：', list(QUERY_TAGS.keys()))

## 4. 批量下载全英 OSM POI（按地区缓存）

分 England / Scotland / Wales / Northern Ireland 共 **4 次** Overpass 请求（替代原来 26,959 次），
结果缓存为 parquet，重新运行时直接读取本地文件。

> 首次下载约需 10–30 分钟（取决于 Overpass 服务器负载），之后每次运行 < 1 分钟。

In [ ]:
from pathlib import Path

# 延长 Overpass 超时（大区域查询需要较长时间）
ox.settings.overpass_settings = '[out:json][timeout:600]'

CACHE_DIR  = Path('cache')
CACHE_DIR.mkdir(exist_ok=True)

UK_REGIONS = ['England', 'Scotland', 'Wales', 'Northern Ireland']
POI_CACHE  = CACHE_DIR / 'uk_pois.parquet'
KEEP_COLS  = list(QUERY_TAGS.keys()) + ['geometry']

if POI_CACHE.exists():
    print('从缓存加载合并 POI 数据...')
    pois = gpd.read_parquet(POI_CACHE)
else:
    region_gdfs = []
    for region in UK_REGIONS:
        rc = CACHE_DIR / f'pois_{region.lower().replace(" ", "_")}.parquet'
        if rc.exists():
            print(f'  {region}: 读缓存')
            gdf = gpd.read_parquet(rc)
        else:
            print(f'  {region}: 下载中（可能需要几分钟）...')
            gdf = ox.features_from_place(region, tags=QUERY_TAGS)
            # 只保留打分所需列，减少内存占用
            cols = [c for c in KEEP_COLS if c in gdf.columns]
            gdf = gdf[cols].reset_index(drop=True)
            gdf.to_parquet(rc)
            print(f'    完成：{len(gdf):,} 个特征')
        region_gdfs.append(gdf)

    pois = pd.concat(region_gdfs, ignore_index=True)
    pois.to_parquet(POI_CACHE)

print(f'POI 总数: {len(pois):,}')
pois.head(3)

## 5. 投影 & 空间连接打分

充电站做 500m 缓冲，与全量 POI 做一次 `sjoin`，然后向量化聚合得分。  
全程 **0 次网络请求**，纯本地计算。

In [ ]:
# ── 1. 充电站：点 → 投影 → 500m 缓冲 ─────────────────────────────────────
stations_gdf = gpd.GeoDataFrame(
    df[['StationID', 'Latitude', 'Longitude']],
    geometry=gpd.points_from_xy(df['Longitude'], df['Latitude']),
    crs='EPSG:4326'
).to_crs('EPSG:27700')

stations_buf = stations_gdf.copy()
stations_buf['geometry'] = stations_buf.geometry.buffer(RADIUS_M)

# ── 2. POI：投影到同一坐标系 ────────────────────────────────────────────────
pois_proj = pois.to_crs('EPSG:27700')
poi_cols  = [c for c in QUERY_TAGS.keys() if c in pois_proj.columns] + ['geometry']

# ── 3. 空间连接：每个 POI → 与之相交的充电站缓冲区 ──────────────────────────
print('空间连接中...')
joined = gpd.sjoin(
    pois_proj[poi_cols],
    stations_buf[['StationID', 'geometry']],
    how='inner',
    predicate='intersects'
)
print(f'连接结果：{len(joined):,} 对 (POI, 充电站)')

# ── 4. 向量化打分 ────────────────────────────────────────────────────────────
# 对每条 (key, value, weight) 规则，统计匹配 POI 数量，乘权重后累加到对应场景
records = []
for scene, tag_list in SCENE_TAGS.items():
    for key, value, weight in tag_list:
        if key not in joined.columns:
            continue
        mask = joined[key].notna() if value is True else (joined[key] == value)
        if not mask.any():
            continue
        contrib = (
            joined.loc[mask]
            .groupby('StationID')
            .size()
            .reset_index(name='count')
        )
        contrib['scene']        = scene
        contrib['contribution'] = contrib['count'] * weight
        records.append(contrib[['StationID', 'scene', 'contribution']])

score_df = (
    pd.concat(records, ignore_index=True)
    .groupby(['StationID', 'scene'])['contribution']
    .sum()
    .unstack(fill_value=0)
    .rename(columns=lambda c: f'score_{c}')
)

# 补全可能缺失的场景列
for scene in SCENE_TAGS:
    col = f'score_{scene}'
    if col not in score_df.columns:
        score_df[col] = 0

# ── 5. 确定最终标签（得分最高的场景；全零 → unknown）────────────────────────
score_cols = [f'score_{s}' for s in SCENE_TAGS]
score_df['label'] = score_df[score_cols].apply(
    lambda row: row.idxmax().replace('score_', '') if row.max() > 0 else 'unknown',
    axis=1
)

# 合并回原始充电站表（未匹配到任何 POI 的站点标记 unknown）
df_labeled = df[['StationID']].merge(score_df.reset_index(), on='StationID', how='left')
df_labeled['label'] = df_labeled['label'].fillna('unknown')
for col in score_cols:
    df_labeled[col] = df_labeled[col].fillna(0).astype(int)

print(df_labeled['label'].value_counts())

## 5. 合并并保存结果

In [ ]:
output_path = '../UK_OCM_stations_labeled.csv'
merge_cols = ['StationID', 'label'] + [f'score_{s}' for s in SCENE_TAGS]
df_out = df.merge(df_labeled[merge_cols], on='StationID', how='left')
df_out.to_csv(output_path, index=False)
print(f'已保存至 {output_path}')
df_out['label'].value_counts()

## 6. 可视化

In [ ]:
import matplotlib.pyplot as plt

LABEL_COLORS = {
    'home':              '#4C72B0',
    'work':              '#DD8452',
    'education':         '#55A868',
    'shopping':          '#C44E52',
    'personal_business': '#8172B2',
    'social':            '#937860',
    'leisure':           '#DA8BC3',
    'holiday':           '#8C8C8C',
    'unknown':           '#CCCCCC',
}

counts = df_out['label'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].pie(
    counts,
    labels=counts.index,
    colors=[LABEL_COLORS.get(l, '#CCCCCC') for l in counts.index],
    autopct='%1.1f%%', startangle=140
)
axes[0].set_title('Label Distribution (NTS 9-class)')

for label, grp in df_out.groupby('label'):
    axes[1].scatter(
        grp['Longitude'], grp['Latitude'],
        c=LABEL_COLORS.get(label, '#CCCCCC'),
        label=label, s=5, alpha=0.6
    )
axes[1].set_title('Charging Stations by Scene Label (UK)')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
axes[1].legend(markerscale=3)

plt.tight_layout()
plt.savefig('label_distribution.png', dpi=150)
plt.show()